# Практика 4. Диагностика выразительности GNN

Сегодня:
1. знакомимся с двумя датасетами-ловушками: CSL и Roman-empire;
2. руками проверяем, что WL не различает графы CSL;
3. собираем классификатор графов на `GINConv` из PyG и убеждаемся, что GCN и GIN на CSL угадывают наугад;
4. добавляем структурную кодировку RWSE и смотрим, как растёт точность;
5. считаем гомофилию и сравниваем MLP, GCN и GraphSAGE на гетерофильном графе.

Всё, что делаем сегодня, — по лекции 4. Слои и кодировки берём готовыми из PyG: сегодня важно
не как они написаны, а **что они могут и чего не могут**.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

try:
    import torch_geometric  # noqa: F401
except ImportError:
    !pip install -q torch_geometric

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn.functional as F            # функции без весов: relu, dropout, cross_entropy
from sklearn.model_selection import StratifiedKFold
from torch import nn                       # слои с весами: Linear, BatchNorm1d, Sequential
from torch_geometric.datasets import GNNBenchmarkDataset, HeterophilousGraphDataset, Planetoid
from torch_geometric.loader import DataLoader   # склеивает много маленьких графов в один батч
from torch_geometric.nn import GCNConv, GINConv, SAGEConv, global_mean_pool
from torch_geometric.transforms import AddRandomWalkPE
from torch_geometric.utils import homophily, to_networkx

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device)

## 0. Знакомство с данными

### CSL: 150 графов, которые WL не различает

CSL (Circular Skip Links) — синтетический датасет. Каждый граф — это цикл из 41 вершины, в котором
каждая вершина дополнительно соединена с вершиной через R шагов по циклу. Число R задаёт класс:
10 разных R — 10 классов, по 15 графов на класс. Графы одного класса отличаются только нумерацией
вершин.

In [ ]:
csl = GNNBenchmarkDataset(root="/tmp/CSL", name="CSL")   # при первом запуске скачается
print(csl)
print("графов:", len(csl), "| классов:", csl.num_classes)

g0 = csl[0]
print("пример графа:", g0)
# ⚠ у графов CSL нет признаков вершин: поля x нет совсем.
# Модель сможет опираться только на структуру — ради этого датасет и придуман
print("признаки вершин:", g0.x)

# edge_index хранит каждое ребро в обе стороны, поэтому степень = сколько раз вершина встретилась
# как источник. bincount считает это для всех вершин сразу
deg = torch.bincount(g0.edge_index[0])
print("степени вершин:", sorted(set(deg.tolist())))      # у всех вершин одна и та же степень
print("классы и число графов:", dict(sorted(Counter(int(g.y) for g in csl).items())))

In [ ]:
def in_cycle_order(g):
    # вершины пронумерованы по циклу, если есть все рёбра 0–1, 1–2, …, 40–0
    edges = set(map(tuple, g.edge_index.t().tolist()))
    return all((i, (i + 1) % g.num_nodes) in edges for i in range(g.num_nodes))

def skip_length(g):
    # работает для графа с нумерацией по циклу. Соседи вершины 0 — это 1 и 40 по циклу
    # плюс две хорды; расстояние по циклу до хорды и есть R
    n = g.num_nodes
    nbrs = g.edge_index[1][g.edge_index[0] == 0].tolist()
    return min(min(u, n - u) for u in nbrs if u not in (1, n - 1))

# ⚠ у большинства графов нумерация вершин перемешана — так графы одного класса и отличаются.
# Первый граф каждого класса записан по порядку цикла: его и берём, чтобы прочитать R и нарисовать
by_class = {}
for g in csl:
    if in_cycle_order(g):
        by_class.setdefault(int(g.y), g)
print("класс → R:", {c: skip_length(g) for c, g in sorted(by_class.items())})

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, c in zip(axes, (0, 1)):
    g = by_class[c]
    n = g.num_nodes
    # вершина i — на окружности под углом 2πi/n: цикл идёт по кругу, хорды видны внутри
    pos = {i: (np.cos(2 * np.pi * i / n), np.sin(2 * np.pi * i / n)) for i in range(n)}
    nx.draw(to_networkx(g, to_undirected=True), pos, ax=ax, node_size=25, width=0.6,
            node_color="#0E8F8F", edge_color="#9AA5B4")
    ax.set_title(f"класс {c}: R = {skip_length(g)}")
plt.show()

### Roman-empire: соседи почти всегда из другого класса

Граф построен по статье английской Википедии «Roman Empire». Вершина — слово текста, ребро
соединяет слова, которые стоят рядом в тексте или связаны в синтаксическом дереве предложения.
Класс вершины — синтаксическая роль слова: подлежащее, определение, обстоятельство и так далее
(17 самых частых ролей, остальные собраны в восемнадцатый класс). Признаки вершины — вектор слова
из fastText.

Соседние слова почти никогда не играют одну и ту же роль — это и есть **гетерофилия**.

In [ ]:
roman = HeterophilousGraphDataset(root="/tmp/het", name="Roman-empire")[0]
print(roman)
print(f"вершин: {roman.num_nodes}, рёбер: {roman.num_edges // 2}, "
      f"признаков: {roman.num_features}, классов: {int(roman.y.max()) + 1}")

# ⚠ маски здесь не векторы, а матрицы [n, 10]: авторы датасета заготовили 10 разных разбиений.
# Столбец k — k-е разбиение. Сегодня берём первые три и усредняем результат
print("форма train_mask:", tuple(roman.train_mask.shape))
print(f"в разбиении 0: train {int(roman.train_mask[:, 0].sum())}, val {int(roman.val_mask[:, 0].sum())}, "
      f"test {int(roman.test_mask[:, 0].sum())}")

# классы сильно несбалансированы — полезно знать, какая точность у «всегда самый частый класс»
counts = torch.bincount(roman.y)
print("размеры классов:", counts.tolist())
print(f"доля самого частого класса: {counts.max() / roman.num_nodes:.3f}")

## 1. WL руками: проверяем, что графы для него одинаковы

Напомним шаг WL из практики 1: новая метка вершины — хеш от её метки и отсортированного списка
меток соседей. Все вершины стартуют с одной меткой: признаков у нас нет.

Если после нескольких итераций **гистограммы меток** двух графов совпали, WL графы не различил.
По теореме из лекции их не различит и ни одна обычная GNN.

In [ ]:
def wl_histogram(graph, iterations=5):
    # graph — граф networkx. Возвращает Counter «метка -> сколько вершин с такой меткой»
    labels = {v: 0 for v in graph.nodes()}          # все вершины стартуют одного цвета
    for _ in range(iterations):
        # ⚠ новые метки собираем в НОВЫЙ словарь: пока считаем шаг, нужны старые метки соседей
        labels = {v: hash((labels[v], tuple(sorted(labels[u] for u in graph.neighbors(v)))))
                  for v in graph.nodes()}
    return Counter(labels.values())

def wl_same(g1, g2, iterations=5):
    return wl_histogram(g1, iterations) == wl_histogram(g2, iterations)

In [ ]:
# пара из лекции: цикл из шести вершин против двух треугольников
c6 = nx.cycle_graph(6)
two_triangles = nx.disjoint_union(nx.cycle_graph(3), nx.cycle_graph(3))
print("C6 и два треугольника: изоморфны?", nx.is_isomorphic(c6, two_triangles),
      "| WL видит одинаково?", wl_same(c6, two_triangles))

# два графа CSL разных классов
a = to_networkx(by_class[0], to_undirected=True)
b = to_networkx(by_class[1], to_undirected=True)
print("CSL, классы 0 и 1: изоморфны?", nx.is_isomorphic(a, b), "| WL видит одинаково?", wl_same(a, b))

# контрольный пример: графы с разными степенями WL обязан различить
print("C6 и путь из 6 вершин: WL видит одинаково?", wl_same(c6, nx.path_graph(6)))

Проверьте себя: почему для CSL хватает одной итерации WL, чтобы понять, что дальше ничего не изменится?
(Подсказка: какая степень у каждой вершины?)

## 2. Классификатор графов на GINConv

Слой GIN из лекции:

$$h_v' = \mathrm{MLP}\Big((1 + \varepsilon)\, h_v + \sum_{u \in \mathcal{N}(v)} h_u\Big)$$

В PyG это `GINConv(mlp)`: сумму по соседям слой считает сам, а MLP мы передаём ему готовым.
По умолчанию ε = 0 и не обучается.

Для классификации **графов** нужен ещё пулинг: из векторов всех вершин графа получить один вектор
графа. `global_mean_pool(h, batch)` усредняет вершины каждого графа в батче. Вектор `batch` говорит,
какому графу принадлежит каждая вершина: `[0, 0, …, 0, 1, 1, …, 1, …]`.

In [ ]:
def gin_layer(hidden):
    # MLP внутри GIN: два линейных слоя. BatchNorm стабилизирует обучение — суммы по соседям
    # бывают крупными, особенно у вершин с большой степенью
    mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                        nn.Linear(hidden, hidden))
    return GINConv(mlp)


class GraphNet(nn.Module):
    def __init__(self, conv="gin", pe_dim=0, hidden=64, layers=4, classes=10):
        super().__init__()
        self.pe_dim = pe_dim
        # кодировку нормируем отдельно: вероятности возврата — маленькие числа, и без нормировки
        # их сигнал теряется рядом с другими входами
        self.pe_norm = nn.BatchNorm1d(pe_dim) if pe_dim else None
        # вход: единица (признаков у CSL нет) плюс кодировка, если она есть
        self.inp = nn.Linear(1 + pe_dim, hidden)
        # ModuleList, а не обычный список: только так веса слоёв видит оптимизатор
        make = (lambda: gin_layer(hidden)) if conv == "gin" else (lambda: GCNConv(hidden, hidden))
        self.convs = nn.ModuleList([make() for _ in range(layers)])
        # голова: из вектора графа — логиты классов
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, classes))

    def forward(self, batch):
        # признаков нет — даём каждой вершине одну и ту же единицу.
        # ⚠ так все вершины стартуют одинаковыми, ровно как в WL
        x = torch.ones(batch.num_nodes, 1, device=batch.edge_index.device)
        if self.pe_dim:
            x = torch.cat([x, self.pe_norm(batch.pe)], dim=1)   # [вершины, 1 + pe_dim]
        h = self.inp(x)
        for conv in self.convs:
            h = F.relu(conv(h, batch.edge_index))   # [все вершины батча, hidden]
        # batch.batch — номер графа для каждой вершины; усредняем вершины каждого графа.
        # Все графы CSL одного размера, так что среднее и сумма отличаются лишь множителем 41
        hg = global_mean_pool(h, batch.batch)       # [графы в батче, hidden]
        return self.head(hg)                     # [графы в батче, classes]


model = GraphNet("gin")
batch = next(iter(DataLoader(csl, batch_size=8)))
print(batch)                                     # 8 графов склеены в один большой граф
print("выход:", model(batch).shape)              # ожидаем [8, 10]

## 3. GCN и GIN на CSL

Датасет маленький, поэтому качество меряем кросс-валидацией: 5 фолдов, в каждом модель учится
на 120 графах и проверяется на 30. Фолды стратифицированы — в каждом поровну графов всех классов.

Кроме точности на тесте печатаем **точность на обучении**. Это главный диагностический приём
лекции 3: если модель не может выучить даже обучающие графы, дело не в переобучении.

In [ ]:
def accuracy(model, graphs):
    model.eval()
    batch = next(iter(DataLoader(graphs, batch_size=len(graphs)))).to(device)   # все графы разом
    with torch.no_grad():
        return float((model(batch).argmax(1) == batch.y).float().mean())


def cross_validate(make_model, graphs, epochs=200, lr=1e-3):
    y = np.array([int(g.y) for g in graphs])
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    train_accs, test_accs = [], []
    # split() возвращает номера графов для обучения и теста; признаки ему не нужны, только метки
    for fold, (tr, te) in enumerate(folds.split(np.zeros(len(y)), y)):
        torch.manual_seed(fold)                   # одинаковые начальные веса при перезапуске
        model = make_model().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        train_graphs, test_graphs = [graphs[i] for i in tr], [graphs[i] for i in te]
        loader = DataLoader(train_graphs, batch_size=16, shuffle=True)
        for _ in range(epochs):
            model.train()
            for batch in loader:
                batch = batch.to(device)
                opt.zero_grad()
                F.cross_entropy(model(batch), batch.y).backward()
                opt.step()
        train_accs.append(accuracy(model, train_graphs))
        test_accs.append(accuracy(model, test_graphs))
    print(f"  train {np.mean(train_accs):.3f} | test {np.mean(test_accs):.3f} ± {np.std(test_accs):.3f}")
    return float(np.mean(test_accs))


results = {}
csl_list = list(csl)                              # список графов: так удобнее брать их по номерам
for name, conv in (("GCN", "gcn"), ("GIN", "gin")):
    print(name)
    results[name] = cross_validate(lambda: GraphNet(conv), csl_list)

⚠ Точность 0.1 при десяти классах — это угадывание. И на обучении тоже 0.1: модель не может
запомнить даже графы, которые видела. Это не плохие гиперпараметры и не мало эпох — для GCN и GIN
все вершины всех графов CSL одинаковы, как и для WL. Выход для всех графов один, сколько эпох ни добавляй.

Убедимся в этом напрямую: посмотрим, сколько разных векторов выдаёт модель на всех 150 графах.

In [ ]:
model = GraphNet("gin").to(device)
model.eval()
with torch.no_grad():
    out = model(next(iter(DataLoader(csl_list, batch_size=len(csl_list)))).to(device))
# округляем, чтобы погрешности float не дали «разные» векторы там, где их нет
print("разных выходов на 150 графах:", len(torch.unique(out.round(decimals=4), dim=0)))

## 4. Подсказываем структуру: RWSE

RWSE вершины v — вектор вероятностей вернуться в v случайным блужданием ровно за 1, 2, …, k шагов.
В PyG его считает трансформ `AddRandomWalkPE(walk_length=k, attr_name="pe")`: он добавляет к графу
поле `pe` формы `[вершины, k]`.

Кодировка считается **один раз до обучения**, сама сеть не меняется — меняется только вход.

In [ ]:
rw = AddRandomWalkPE(walk_length=20, attr_name="pe")
csl_rw = [rw(g.clone()) for g in csl]               # clone — чтобы не менять исходный датасет
print(csl_rw[0])                                     # появилось поле pe=[41, 20]

In [ ]:
# Сравним кодировки вершины 0 в графах двух разных классов
rw_by_class = {}
for g in csl_rw:
    if in_cycle_order(g):                           # те же графы, что мы рисовали выше
        rw_by_class.setdefault(int(g.y), g)
np.set_printoptions(precision=3, suppress=True)
for c in (0, 1):
    print(f"класс {c}, R = {skip_length(rw_by_class[c])}:", rw_by_class[c].pe[0, :8].numpy())

# ⚠ внутри одного графа CSL все вершины симметричны, поэтому RWSE у всех вершин графа одинаковый.
# Различает он не вершины, а графы: у разных R блуждание возвращается с разной вероятностью.
# Сравниваем через allclose: при подсчёте накапливается погрешность float порядка 1e-8,
# и проверка «==» сказала бы «разные» там, где разница — шум
pe = csl_rw[0].pe
print("одинаковы ли RWSE всех вершин графа:", torch.allclose(pe, pe[0].expand_as(pe), atol=1e-6))

Уже на шагах 3–4 векторы разных классов расходятся: хорды создают короткие циклы, и блуждание
возвращается домой разными путями. WL этого не видит — у всех вершин просто степень 4.

In [ ]:
print("GIN + RWSE")
results["GIN + RWSE"] = cross_validate(lambda: GraphNet("gin", pe_dim=20), csl_rw)

for name, acc in results.items():
    print(f"{name:12s} {acc:.3f}")

## 5. Гетерофилия: когда соседи мешают

**Гомофилия по рёбрам** — доля рёбер, у которых оба конца одного класса. Посчитаем её для Cora
и Roman-empire.

In [ ]:
def edge_homophily(edge_index, y):
    # edge_index: [2, E], y: [n] — метки вершин
    same = y[edge_index[0]] == y[edge_index[1]]     # [E] — True, если концы ребра одного класса
    # ⚠ каждое ребро записано в обе стороны, но это не мешает: обе копии дают одинаковый ответ,
    # и доля от этого не меняется
    return float(same.float().mean())

cora = Planetoid(root="/tmp/Cora", name="Cora")[0]
for name, g in (("Cora", cora), ("Roman-empire", roman)):
    mine = edge_homophily(g.edge_index, g.y)
    # сверяемся с готовой функцией PyG — должно совпасть
    print(f"{name:13s} {mine:.3f}   PyG: {homophily(g.edge_index, g.y, method='edge'):.3f}")

В Cora 81% рёбер соединяют статьи одной области. В Roman-empire — 5%: соседнее слово почти
всегда играет другую роль. Сравним три модели:

- **MLP** — граф не видит вовсе, только вектор слова;
- **GCN** — усредняет вершину вместе с соседями, свой вектор смешан с чужими;
- **GraphSAGE** — `W₁ h_v + W₂ · среднее(h_u)`: свой вектор и соседей обрабатывают разные веса.

In [ ]:
class NodeNet(nn.Module):
    def __init__(self, kind, in_dim, hidden, classes):
        super().__init__()
        self.kind = kind
        layer = {"mlp": nn.Linear, "gcn": GCNConv, "sage": SAGEConv}[kind]
        self.l1 = layer(in_dim, hidden)
        self.l2 = layer(hidden, classes)

    def call(self, layer, h, edge_index):
        # ⚠ nn.Linear принимает только h, графовые слои — ещё и edge_index
        return layer(h) if self.kind == "mlp" else layer(h, edge_index)

    def forward(self, x, edge_index):
        h = F.relu(self.call(self.l1, x, edge_index))
        h = F.dropout(h, p=0.5, training=self.training)
        return self.call(self.l2, h, edge_index)

In [ ]:
def train_node(kind, g, split, epochs=300, hidden=256, lr=3e-3):
    tr, va, te = g.train_mask[:, split], g.val_mask[:, split], g.test_mask[:, split]
    torch.manual_seed(split)
    model = NodeNet(kind, g.num_features, hidden, int(g.y.max()) + 1).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_val, best_test = 0.0, 0.0
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        F.cross_entropy(model(g.x, g.edge_index)[tr], g.y[tr]).backward()
        opt.step()
        if epoch % 5 == 0:                        # проверяем не каждую эпоху — так быстрее
            model.eval()
            with torch.no_grad():
                pred = model(g.x, g.edge_index).argmax(1)
            val_acc = float((pred[va] == g.y[va]).float().mean())
            # ранняя остановка: запоминаем тест в момент лучшей валидации, а не лучший тест
            if val_acc > best_val:
                best_val, best_test = val_acc, float((pred[te] == g.y[te]).float().mean())
    return best_test


roman = roman.to(device)
node_results = {}
for name, kind in (("MLP без графа", "mlp"), ("GCN", "gcn"), ("GraphSAGE", "sage")):
    accs = [train_node(kind, roman, split) for split in range(3)]
    node_results[name] = accs
    print(f"{name:14s} {np.mean(accs):.3f} ± {np.std(accs):.3f}")

In [ ]:
names = list(node_results)
means = [np.mean(node_results[n]) for n in names]
stds = [np.std(node_results[n]) for n in names]
plt.figure(figsize=(6, 3.5))
plt.bar(names, means, yerr=stds, capsize=4, color=["#6B7280", "#E07A3F", "#0E8F8F"])
plt.ylabel("accuracy на тесте")
plt.title("Roman-empire, 3 разбиения")
plt.ylim(0, 1)
plt.show()

GCN проигрывает даже MLP, который граф вообще не видит: усреднение смешивает слово с соседями,
у которых почти всегда другая роль. GraphSAGE держит свой вектор отдельно и выигрывает у обоих —
граф полезен, если не растворять в нём собственный сигнал вершины.

## 6. Выводы

- Обычная GNN различает не больше графов, чем WL. На CSL GCN и GIN не выучивают даже обучающие
  графы — это предел выразительности, а не плохое обучение.
- Структурная кодировка на входе снимает этот предел для CSL почти даром: сеть та же, меняется
  только вход.
- Гомофилия — первое, что стоит посчитать на новом графе. При сильной гетерофилии слой должен
  держать собственный вектор вершины отдельно от соседей, а MLP — обязательный baseline.

### Задания для самостоятельной работы
1. Замените RWSE на `AddLaplacianEigenvectorPE(k=8, attr_name="pe", is_undirected=True)`.
   ⚠ Знак собственного вектора произволен: при обучении умножайте каждый столбец `pe` на
   случайные ±1. Что будет, если этого не делать?
2. Уменьшайте `walk_length` у RWSE: 2, 3, 4, 6, 10. С какой длины точность на CSL заметно выше
   угадывания и почему?
3. GIN + RWSE и на обучении даёт не 1.0. Постройте матрицу ошибок
   (`sklearn.metrics.confusion_matrix`): какие классы модель путает? Сравните их RWSE.
4. Добавьте к GCN отдельную линейную проекцию собственного вектора: `GCNConv(h) + Linear(h)`.
   Догонит ли она GraphSAGE на Roman-empire?
5. Посчитайте гомофилию Amazon-ratings (`HeterophilousGraphDataset(name="Amazon-ratings")`)
   и сравните на нём те же три модели. Как меняется разрыв между ними?